# CONNECT DRIVE

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# RANDOM FOREST MODEL — Multi-Seed (mean ± std)

## Import & Config

In [2]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEEDS        = [42, 0, 1, 2, 3]   # 5 independent runs
N_ESTIMATORS = 500
MAX_DEPTH    = 16
BASE_PATH    = '/content/drive/MyDrive/wind_forecast/'
SAVE_DIR     = '/content/drive/MyDrive/wind_forecast/RF_Results/'
os.makedirs(SAVE_DIR, exist_ok=True)

STATIONS = [
    ('Site A - Inland',          'A'),
    ('Site B - Coastal',         'B'),
    ('Site C - Complex Terrain', 'C'),
    ('Site D - Offshore',        'D'),
]
HORIZONS = ['30min', '60min', '120min']

print(f'Seeds       : {SEEDS}')
print(f'n_estimators: {N_ESTIMATORS}')
print(f'max_depth   : {MAX_DEPTH}')

Seeds       : [42, 0, 1, 2, 3]
n_estimators: 500
max_depth   : 16


## Data Loading

In [3]:
def load_station_data(file_path, station_name):
    print(f"  Loading: {station_name}")
    df = pd.read_csv(file_path)
    df['time'] = pd.to_datetime(df['time'])
    df.set_index('time', inplace=True)

    # Targets: 1 step=15 min → +2=30min, +4=60min, +8=120min
    df['target_30min']  = df['wind_speed'].shift(-2)
    df['target_60min']  = df['wind_speed'].shift(-4)
    df['target_120min'] = df['wind_speed'].shift(-8)
    df = df.dropna()

    target_cols = ['target_30min', 'target_60min', 'target_120min']
    Y = df[target_cols].values
    X = df.drop(columns=target_cols).values

    n         = len(df)
    train_end = int(n * 0.70)
    val_end   = int(n * 0.85)

    X_train, Y_train = X[:train_end],       Y[:train_end]
    X_val,   Y_val   = X[train_end:val_end], Y[train_end:val_end]
    X_test,  Y_test  = X[val_end:],          Y[val_end:]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)

    print(f"Train={len(X_train):,} | Val={len(X_val):,} | Test={len(X_test):,}")
    return X_train, Y_train, X_val, Y_val, X_test, Y_test

## Training & Evaluation (Multi-Seed)

In [4]:
def run_one_seed(X_train, Y_train, X_test, Y_test, seed):
    rf = RandomForestRegressor(
        n_estimators=N_ESTIMATORS,
        max_depth=MAX_DEPTH,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=seed,
        n_jobs=-1
    )

    t0 = time.time()
    rf.fit(X_train, Y_train)
    train_time = time.time() - t0

    # Latency
    t0     = time.time()
    y_pred = rf.predict(X_test)
    latency_ms = (time.time() - t0) / len(X_test) * 1000

    metrics = {'train_time_s': train_time, 'latency_ms': latency_ms}
    for i, hor in enumerate(HORIZONS):
        metrics[f'RMSE_{hor}'] = float(np.sqrt(mean_squared_error(Y_test[:, i], y_pred[:, i])))
        metrics[f'MAE_{hor}']  = float(mean_absolute_error(Y_test[:, i], y_pred[:, i]))
        metrics[f'R2_{hor}']   = float(r2_score(Y_test[:, i], y_pred[:, i]))

    # Global RMSE
    metrics['RMSE_Global'] = float(np.sqrt(mean_squared_error(Y_test.ravel(), y_pred.ravel())))
    metrics['MAE_Global']  = float(mean_absolute_error(Y_test.ravel(), y_pred.ravel()))
    metrics['R2_Global']   = float(r2_score(Y_test.ravel(), y_pred.ravel()))

    return metrics, rf


def fmt(mean, std):
    return f"{mean:.4f}±{std:.4f}"


def aggregate(run_list):
    keys  = list(run_list[0].keys())
    agg   = {}
    for k in keys:
        vals = [r[k] for r in run_list]
        agg[k] = (float(np.mean(vals)), float(np.std(vals)))
    return agg

In [5]:
all_results  = {}
all_raw      = {}
last_models  = {}

total_start = time.time()

for station_name, code in STATIONS:
    file_path = f"{BASE_PATH}{station_name}.csv"
    print(f"Station: {station_name}")

    X_tr, Y_tr, X_va, Y_va, X_te, Y_te = load_station_data(file_path, station_name)

    run_metrics = []
    for seed in SEEDS:
        print(f" Seed {seed}...", end=' ')
        m, rf_model = run_one_seed(X_tr, Y_tr, X_te, Y_te, seed)
        run_metrics.append(m)
        print(f"RMSE_30min={m['RMSE_30min']:.4f} | train={m['train_time_s']:.0f}s")

    agg = aggregate(run_metrics)
    all_results[code] = agg
    all_raw[code]     = run_metrics
    last_models[code] = rf_model

    # ── Print station summary ──
    print(f"\nResults for {station_name} (mean±std over {len(SEEDS)} seeds):")
    print(f"{'Horizon':<12} {'RMSE':>22} {'MAE':>22} {'R2':>22}")
    for hor in HORIZONS + ['Global']:
        rmse_str = fmt(*agg[f'RMSE_{hor}'])
        mae_str  = fmt(*agg[f'MAE_{hor}'])
        r2_str   = fmt(*agg[f'R2_{hor}'])
        print(f"  {hor:<12} {rmse_str:>22} {mae_str:>22} {r2_str:>22}")
    print(f"Train time : {fmt(*agg['train_time_s'])} s")
    print(f"Latency    : {fmt(*agg['latency_ms'])} ms/sample")

total_time = time.time() - total_start
print(f"Total pipeline time: {total_time/60:.1f} min")

Station: Site A - Inland
  Loading: Site A - Inland
Train=73,626 | Val=15,777 | Test=15,778
 Seed 42... RMSE_30min=0.2388 | train=229s
 Seed 0... RMSE_30min=0.2417 | train=232s
 Seed 1... RMSE_30min=0.2459 | train=225s
 Seed 2... RMSE_30min=0.2324 | train=225s
 Seed 3... RMSE_30min=0.2435 | train=237s

Results for Site A - Inland (mean±std over 5 seeds):
Horizon                        RMSE                    MAE                     R2
  30min                 0.2405±0.0046          0.1348±0.0009          0.9856±0.0006
  60min                 0.4203±0.0028          0.2775±0.0008          0.9560±0.0006
  120min                0.7304±0.0014          0.5391±0.0007          0.8672±0.0005
  Global                0.5060±0.0022          0.3171±0.0008          0.9363±0.0005
Train time : 229.5781±4.5171 s
Latency    : 0.0728±0.0017 ms/sample
Station: Site B - Coastal
  Loading: Site B - Coastal
Train=73,626 | Val=15,777 | Test=15,778
 Seed 42... RMSE_30min=0.1887 | train=223s
 Seed 0... RMSE_30mi

## Paper-Ready Summary Table

In [6]:
rows = []
for station_name, code in STATIONS:
    agg = all_results[code]
    for hor in HORIZONS:
        rows.append({
            'Model'          : 'RF',
            'Site'           : f"{code} ({station_name.split(' - ')[1]})",
            'Horizon'        : hor,
            'RMSE'           : fmt(*agg[f'RMSE_{hor}']),
            'MAE'            : fmt(*agg[f'MAE_{hor}']),
            'R2'             : fmt(*agg[f'R2_{hor}']),
            'RMSE_mean'      : agg[f'RMSE_{hor}'][0],
            'R2_mean'        : agg[f'R2_{hor}'][0],
        })

df_paper = pd.DataFrame(rows)

print(df_paper[['Site','Horizon','RMSE','MAE','R2']].to_string(index=False))

print('\nAveraged across all 4 sites:')
for hor in HORIZONS + ['Global']:
    rmse_vals = [all_results[c][f'RMSE_{hor}'][0] for _, c in STATIONS]
    r2_vals   = [all_results[c][f'R2_{hor}'][0]   for _, c in STATIONS]
    print(f"  {hor}: RMSE = {np.mean(rmse_vals):.4f} | R2 = {np.mean(r2_vals):.4f}")

print('\nComputational Benchmark:')
train_means = [all_results[c]['train_time_s'][0] for _, c in STATIONS]
lat_means   = [all_results[c]['latency_ms'][0]   for _, c in STATIONS]
print(f"  Avg train time : {np.mean(train_means):.2f} s")
print(f"  Avg latency    : {np.mean(lat_means):.4f} ms/sample")

# Save
df_paper.to_csv(f"{SAVE_DIR}rf_results_multiseed.csv", index=False)
print(f"\nSaved → {SAVE_DIR}rf_results_multiseed.csv")

               Site Horizon          RMSE           MAE            R2
         A (Inland)   30min 0.2405±0.0046 0.1348±0.0009 0.9856±0.0006
         A (Inland)   60min 0.4203±0.0028 0.2775±0.0008 0.9560±0.0006
         A (Inland)  120min 0.7304±0.0014 0.5391±0.0007 0.8672±0.0005
        B (Coastal)   30min 0.1887±0.0007 0.1242±0.0004 0.9959±0.0000
        B (Coastal)   60min 0.3481±0.0004 0.2472±0.0003 0.9862±0.0000
        B (Coastal)  120min 0.6027±0.0002 0.4552±0.0002 0.9585±0.0000
C (Complex Terrain)   30min 0.2636±0.0017 0.1498±0.0004 0.9905±0.0001
C (Complex Terrain)   60min 0.4551±0.0009 0.3006±0.0002 0.9718±0.0001
C (Complex Terrain)  120min 0.7461±0.0004 0.5415±0.0001 0.9241±0.0001
       D (Offshore)   30min 0.3334±0.0040 0.1537±0.0008 0.9933±0.0002
       D (Offshore)   60min 0.4929±0.0020 0.2914±0.0005 0.9854±0.0001
       D (Offshore)  120min 0.8320±0.0008 0.5683±0.0004 0.9583±0.0001

Averaged across all 4 sites:
  30min: RMSE = 0.2565 | R2 = 0.9913
  60min: RMSE = 0.4291 

In [7]:
print("TABLE 4 — Random Forest  |  mean +/- std, 5 seeds, averaged 4 sites")
print(f"  {'Horizon':<10}  {'RMSE':>20}  {'MAE':>20}  {'R2':>20}")
for hor in HORIZONS:
    # Per-seed averages across sites
    seed_rmse = [np.mean([all_raw[c][i][f'RMSE_{hor}'] for _, c in STATIONS])
                 for i in range(len(SEEDS))]
    seed_mae  = [np.mean([all_raw[c][i][f'MAE_{hor}']  for _, c in STATIONS])
                 for i in range(len(SEEDS))]
    seed_r2   = [np.mean([all_raw[c][i][f'R2_{hor}']   for _, c in STATIONS])
                 for i in range(len(SEEDS))]
    print(f"  {hor:<10}  {np.mean(seed_rmse):.4f} ± {np.std(seed_rmse):.4f}"
          f"  {np.mean(seed_mae):.4f} ± {np.std(seed_mae):.4f}"
          f"  {np.mean(seed_r2):.4f} ± {np.std(seed_r2):.4f}")

print("TABLE 5 — Random Forest  |  site-wise 30-min  (mean +/- std, 5 seeds)")
_site_names = {'A': 'A (Inland)', 'B': 'B (Coastal)',
               'C': 'C (Complex Terrain)', 'D': 'D (Offshore)'}
print(f"  {'Site':<24}  {'RMSE':>22}  {'R2':>22}")
for _, c in STATIONS:
    rmse_m, rmse_s = all_results[c]['RMSE_30min']
    r2_m,   r2_s   = all_results[c]['R2_30min']
    print(f"  {_site_names[c]:<24}  {rmse_m:.4f} ± {rmse_s:.4f}"
          f"  {r2_m:.4f} ± {r2_s:.4f}")

_all_lat   = [m['latency_ms']   for _, c in STATIONS for m in all_raw[c]]
_all_train = [m['train_time_s'] for _, c in STATIONS for m in all_raw[c]]
lat_mean_rf  = float(np.mean(_all_lat))
lat_std_rf   = float(np.std(_all_lat))
tr_mean_rf   = float(np.mean(_all_train))
tr_std_rf    = float(np.std(_all_train))

_global_rmse_rf = float(np.mean([
    all_raw[c][i][f'RMSE_{h}']
    for _, c in STATIONS for h in HORIZONS for i in range(len(SEEDS))
]))

print("TABLE 6 — Random Forest  |  Model complexity & operational deployment")
print(f"  Parameters        : N/A  ({N_ESTIMATORS} decision trees, ensemble model)")
print(f"  Model footprint   : N/A  (serialized tree structure, size varies with depth)")
print(f"  Latency (CPU)     : {lat_mean_rf:.4f} ± {lat_std_rf:.4f} ms/sample  "
      f"(batch inference / n_test samples)")
print(f"  Global RMSE       : {_global_rmse_rf:.4f}  (avg all sites x horizons x seeds)")
print(f"  Avg train time    : {tr_mean_rf:.2f} ± {tr_std_rf:.2f} s/seed")
print(f"  Total pipeline time: {total_time:.1f} s  ({total_time/60:.2f} min)")

TABLE 4 — Random Forest  |  mean +/- std, 5 seeds, averaged 4 sites
  Horizon                     RMSE                   MAE                    R2
  30min       0.2565 +/- 0.0009  0.1406 +/- 0.0002  0.9913 +/- 0.0001
  60min       0.4291 +/- 0.0006  0.2792 +/- 0.0002  0.9748 +/- 0.0001
  120min      0.7278 +/- 0.0004  0.5260 +/- 0.0002  0.9270 +/- 0.0001
TABLE 5 — Random Forest  |  site-wise 30-min  (mean +/- std, 5 seeds)
  Site                                        RMSE                      R2
  A (Inland)                0.2405 +/- 0.0046  0.9856 +/- 0.0006
  B (Coastal)               0.1887 +/- 0.0007  0.9959 +/- 0.0000
  C (Complex Terrain)       0.2636 +/- 0.0017  0.9905 +/- 0.0001
  D (Offshore)              0.3334 +/- 0.0040  0.9933 +/- 0.0002
TABLE 6 — Random Forest  |  Model complexity & operational deployment
  Parameters        : N/A  (500 decision trees, ensemble model)
  Model footprint   : N/A  (serialized tree structure, size varies with depth)
  Latency (CPU)     : 0.0

In [8]:
rmse_rf_per_seed = {}
for _, c in STATIONS:
    rmse_rf_per_seed[c] = {}
    for hor in HORIZONS:
        rmse_rf_per_seed[c][hor] = [m[f'RMSE_{hor}'] for m in all_raw[c]]

np.save('/content/drive/MyDrive/wind_forecast/rmse_rf_per_seed.npy', rmse_rf_per_seed)
print("Saved → rmse_rf_per_seed.npy")
print("Structure: {site_code: {horizon: [5 values]}}")
for _, c in STATIONS:
    vals_30 = rmse_rf_per_seed[c]['30min']
    print(f"  Site {c} 30min: mean={np.mean(vals_30):.4f}  seeds={[f'{v:.4f}' for v in vals_30]}")

Saved → rmse_rf_per_seed.npy
Structure: {site_code: {horizon: [5 values]}}
  Site A 30min: mean=0.2405  seeds=['0.2388', '0.2417', '0.2459', '0.2324', '0.2435']
  Site B 30min: mean=0.1887  seeds=['0.1887', '0.1879', '0.1880', '0.1891', '0.1896']
  Site C 30min: mean=0.2636  seeds=['0.2609', '0.2624', '0.2656', '0.2647', '0.2644']
  Site D 30min: mean=0.3334  seeds=['0.3335', '0.3381', '0.3310', '0.3372', '0.3271']
